# CareLoop Clinics: No-Show Guard
## Part 3: Machine Learning (predict no-shows and choose the model honestly)

**Input:** `appointments_clean.csv` from Part 1 (rows kept in file order, never shuffled or re-sorted).
**Output:** `test_risk_scores.csv`. (In Colab, upload `appointments_clean.csv` next to this notebook first.)

**Prediction point:** the score is produced the evening before the appointment, after that day's reminders were sent or skipped, so every feature (including `reminder_sent` and `prior_no_shows`) is known at that moment.

**Rule for this Part:** the test set is for reporting only. Model, settings and threshold are all chosen from the training set (cross-validation).
Only about 1 booking in 5 is a no-show, so every model is judged on recall, precision and ROC-AUC as well as accuracy.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.9.1 | pandas 3.0.5 | numpy 2.5.3


## Task 3.1: Features, split and baseline

In [2]:
df = pd.read_csv("appointments_clean.csv")      # do not shuffle or re-sort
ids = df["appointment_id"]                       # kept aside for test_risk_scores.csv

X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

print("X columns (", X.shape[1], "):", list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("No-shows in test set:", int(y_test.sum()), "| overall no-show rate:", round(100 * y.mean(), 1), "%")
baseline = accuracy_score(y_test, np.zeros(len(y_test), dtype=int))
print(f"Baseline accuracy (always 'Attended'): {baseline:.4f} ({int((y_test == 0).sum())} of {len(y_test)})")

X columns ( 11 ): ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Training rows: 792 | Test rows: 198
No-shows in test set: 49 | overall no-show rate: 22.0 %
Baseline accuracy (always 'Attended'): 0.7525 (149 of 198)


## Task 3.2: Logistic regression
The scaler is fitted on the training set only.

In [3]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

def report(name, y_true, pred, prob):
    print(f"{name}")
    print(f"  accuracy  {accuracy_score(y_true, pred):.3f}")
    print(f"  precision {precision_score(y_true, pred, zero_division=0):.3f}")
    print(f"  recall    {recall_score(y_true, pred):.3f}")
    print(f"  F1        {f1_score(y_true, pred):.3f}")
    print(f"  ROC-AUC   {roc_auc_score(y_true, prob):.3f}")

report("Logistic regression (test set, threshold 0.5)", y_test, lr_pred, lr_prob)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, lr_pred))

Logistic regression (test set, threshold 0.5)
  accuracy  0.788
  precision 0.630
  recall    0.347
  F1        0.447
  ROC-AUC   0.789
Confusion matrix [[TN, FP], [FN, TP]]:
[[139  10]
 [ 32  17]]


In [4]:
coefs = pd.DataFrame({"feature": X.columns, "coef (scaled)": lr.coef_[0],
                      "feature_sd (scaler.scale_)": scaler.scale_})
print(coefs.round(3).to_string(index=False))

i = list(X.columns).index("reminder_sent")
c, sd = lr.coef_[0][i], scaler.scale_[i]
per_sd = np.exp(c)
per_unit = np.exp(c / sd)
print()
print(f"reminder_sent coefficient = {c:.3f}, SD of reminder_sent = {sd:.3f}")
print(f"np.exp(coef)      = {per_sd:.2f}  -> odds multiplier per ONE STANDARD DEVIATION of the scaled feature")
print(f"np.exp(coef / SD) = {per_unit:.2f}  -> odds multiplier for a reminder going from No (0) to Yes (1)")
print(f"i.e. about {round(100 * (1 - per_unit))}% lower odds of a no-show with a reminder (association, not cause)")

                feature  coef (scaled)  feature_sd (scaler.scale_)
                    age         -0.254                      16.656
            distance_km          0.253                       6.052
                fee_inr          0.453                     184.254
              lead_days          0.614                       8.614
         prior_no_shows          0.650                       0.969
          reminder_sent         -0.581                       0.495
    booking_channel_App         -0.047                       0.486
  booking_channel_Phone          0.216                       0.481
booking_channel_Walk-In         -0.186                       0.435
visit_type_Consultation          0.016                       0.493
   visit_type_Follow-up         -0.016                       0.493

reminder_sent coefficient = -0.581, SD of reminder_sent = 0.495
np.exp(coef)      = 0.56  -> odds multiplier per ONE STANDARD DEVIATION of the scaled feature
np.exp(coef / SD) = 0.31  -> odds mul

**Why the two numbers differ:** the model was fitted on standardised features, so the raw coefficient describes a change of one standard deviation (about 0.49) in `reminder_sent`, while a real No→Yes change is a full unit (about two SDs). Dividing by `scaler.scale_` converts the coefficient to that one-unit change, which gives the stronger effect.

**Conclusion (association, not cause):** in this model, holding the other features constant, appointments with a reminder have roughly 0.31 times the odds of a no-show (about 69% lower odds). `reminder_sent` has the most negative coefficient, while `lead_days` and `prior_no_shows` have the largest positive ones. The data are observational, so this does not prove that sending reminders *causes* fewer no-shows. For example, reminders may go more often to a kind of patient who attends anyway.

*(The exact numbers are in the output above; this text is checked against them in the final cell.)*

## Task 3.3: Decision tree and overfitting

In [5]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

for name, m in [("Full-depth tree", tree_full), ("Depth-3 tree", tree3)]:
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"{name}: depth={m.get_depth()}, training accuracy={tr:.3f}, test accuracy={te:.3f}, gap={tr - te:.3f}")
print(f"(baseline accuracy = {baseline:.4f})")

Full-depth tree: depth=15, training accuracy=1.000, test accuracy=0.672, gap=0.328
Depth-3 tree: depth=3, training accuracy=0.822, test accuracy=0.778, gap=0.044
(baseline accuracy = 0.7525)


**What the gap shows:** the unlimited tree scores 100% on training data but below the 'always Attended' baseline on the test set, so it memorised noise in the training rows (overfitting). Limiting the depth to 3 gives a much smaller gap and better test accuracy, so it generalises better.

## Task 3.4: Random forest and a comparison table

In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print(f"Random forest: training accuracy={accuracy_score(y_train, rf.predict(X_train)):.3f}")
report("Random forest (test set, threshold 0.5)", y_test, rf_pred, rf_prob)

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop three feature importances:")
print(imp.head(3).round(3).to_string())

Random forest: training accuracy=1.000
Random forest (test set, threshold 0.5)
  accuracy  0.747
  precision 0.484
  recall    0.306
  F1        0.375
  ROC-AUC   0.706

Top three feature importances:
lead_days      0.220
age            0.197
distance_km    0.195


In [7]:
t3_pred = tree3.predict(X_test)
t3_prob = tree3.predict_proba(X_test)[:, 1]
comparison = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, p) for p in (lr_pred, t3_pred, rf_pred)],
    "recall": [recall_score(y_test, p) for p in (lr_pred, t3_pred, rf_pred)],
    "ROC-AUC": [roc_auc_score(y_test, p) for p in (lr_prob, t3_prob, rf_prob)],
}).round(3)
print("Test-set comparison (for reporting only, not for choosing):")
print(comparison.to_string(index=False))

Test-set comparison (for reporting only, not for choosing):
                  model  accuracy  recall  ROC-AUC
    Logistic regression     0.788   0.347    0.789
Decision tree (depth 3)     0.778   0.224    0.700
          Random forest     0.747   0.306    0.706


**Which looks best:** logistic regression has the highest ROC-AUC and the highest recall, so it ranks risky bookings best and catches the most no-shows. The random forest's perfect training accuracy against weaker test numbers points to overfitting on a small dataset with mostly linear effects. This table is for reporting only. The formal choice is made in Task 3.5 from training data alone.

## Task 3.5: Tune, choose the model and the threshold, then test once
### (a) Grid search for the random forest (training set only)

In [8]:
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print(f"best_score_ (cross-validated ROC-AUC): {grid.best_score_:.3f}")

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (cross-validated ROC-AUC): 0.767


### (b) Choose the final model from cross-validated training ROC-AUC
Logistic regression uses the supplied pipeline on the **unscaled** `X_train`, so the scaler is re-fitted inside each fold.

In [9]:
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

cv_scores = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (100 trees)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    f"Tuned random forest {grid.best_params_}": grid.best_score_,
}
cv_table = pd.Series(cv_scores, name="CV ROC-AUC (train)").round(3).sort_values(ascending=False)
print(cv_table.to_string())
winner = cv_table.index[0]
print("\nWinner:", winner)

Logistic regression (pipeline)                               0.774
Tuned random forest {'max_depth': 3, 'n_estimators': 200}    0.767
Random forest (100 trees)                                    0.747
Decision tree (depth 3)                                      0.678

Winner: Logistic regression (pipeline)


### (c) Choose the threshold from out-of-fold training probabilities

In [10]:
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in (0.5, 0.35):
    p = (oof >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_train, p),
                 "recall": recall_score(y_train, p),
                 "F1": f1_score(y_train, p),
                 "flagged": int(p.sum())})
thr_table = pd.DataFrame(rows).round(3)
print("Out-of-fold results on the training set (792 rows):")
print(thr_table.to_string(index=False))
THRESHOLD = float(thr_table.loc[thr_table["F1"].idxmax(), "threshold"])
print("\nChosen threshold (higher F1):", THRESHOLD)

Out-of-fold results on the training set (792 rows):
 threshold  precision  recall    F1  flagged
      0.50      0.575   0.249 0.347       73
      0.35      0.513   0.467 0.489      154

Chosen threshold (higher F1): 0.35


**Why recall matters here:** a missed warning (false negative) costs an empty doctor slot, while an unnecessary call (false positive) costs about a minute of front-desk time. So catching more real no-shows is worth some extra calls. The lower threshold raises recall a lot for a small loss of precision, and it also has the higher F1.

### (d) Fit the winner on the full training set and evaluate **once** on the test set

In [11]:
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flag = (probs >= THRESHOLD).astype(int)

final = {"precision": precision_score(y_test, flag), "recall": recall_score(y_test, flag),
         "flagged": int(flag.sum()), "ROC-AUC": roc_auc_score(y_test, probs)}
print(f"Final model: logistic regression (pipeline), threshold {THRESHOLD}, evaluated once on the test set")
for k, v in final.items():
    print(f"  {k:9s} {v:.3f}" if isinstance(v, float) else f"  {k:9s} {v}")
print("  confusion matrix:", confusion_matrix(y_test, flag).tolist())
print(f"  rows within 0.002 of the threshold: {int((np.abs(probs - THRESHOLD) < 0.002).sum())}")
print(f"(For information only) tuned forest test ROC-AUC: {roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1]):.3f}")

Final model: logistic regression (pipeline), threshold 0.35, evaluated once on the test set
  precision 0.571
  recall    0.571
  flagged   49
  ROC-AUC   0.789
  confusion matrix: [[128, 21], [21, 28]]
  rows within 0.002 of the threshold: 1
(For information only) tuned forest test ROC-AUC: 0.741


### (e) Save the test-set risk scores

In [12]:
scores = pd.DataFrame({"appointment_id": ids.loc[X_test.index].values,
                       "no_show": y_test.values,
                       "risk_prob": probs.round(4),
                       "flag": flag})
scores.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv:", scores.shape, "| rows with flag = 1:", int(scores["flag"].sum()))
print(scores.sort_values("risk_prob", ascending=False).head(10).to_string(index=False))

Saved test_risk_scores.csv: (198, 4) | rows with flag = 1: 49
appointment_id  no_show  risk_prob  flag
         A0635        1     0.9372     1
         A0544        1     0.9241     1
         A0293        1     0.9196     1
         A0950        1     0.8186     1
         A0996        0     0.7667     1
         A0067        0     0.7498     1
         A0769        1     0.7444     1
         A0364        0     0.7319     1
         A0806        1     0.7035     1
         A0560        0     0.7019     1


**How a clinic would really choose the threshold:** start from how many follow-up calls the front desk can make in an evening, set the threshold so the number flagged fits that capacity, and weigh it against the cost of an empty slot. If calls are cheap and slots are expensive, move the threshold lower.

**Honest limits (also in the README):** the Task 1.3 medians, quartiles and fences were computed on all 990 rows before the split; the same patient can appear in both train and test; the split is random, not past-to-future; and there is only one train/test split.